# Drama LOB -- Google Colab Pretraining

Pretrain the Drama world model on Polymarket limit-order-book features. Replaces the CNN encoder/decoder with a Transformer over depth-curve tokens.

**Before running:**
1. `Runtime -> Change runtime type -> A100 GPU` (Colab Pro).
2. Push your LOB branch to your fork (`git push origin <branch>`). Set `REPO_URL` and `BRANCH` in cell 3.
3. Keep `drive-download-20260423T214219Z-3-001.zip` anywhere under `/content/drive/MyDrive/` -- cell 2 auto-discovers it. If the zip lives in a shared folder, right-click the folder in Drive -> Organize -> Add shortcut -> MyDrive.

In [ ]:
# Step 1 -- Verify GPU
!nvidia-smi

: 

In [ ]:
# Step 2 -- Mount Google Drive and locate the LOB data zip
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/drama_lob_checkpoints'
!mkdir -p {DRIVE_CKPT}

# Auto-locate the zip anywhere under MyDrive. If you have multiple matches,
# set DRIVE_DATA_ZIP explicitly to the one you want.
import glob, os
candidates = (
    glob.glob('/content/drive/MyDrive/**/drive-download-*.zip', recursive=True)
    + glob.glob('/content/drive/MyDrive/**/*polymarket*.zip', recursive=True)
    + glob.glob('/content/drive/MyDrive/**/*orderbook*.zip', recursive=True)
)
DRIVE_DATA_ZIP = candidates[0] if candidates else ''
print(f'Checkpoint backup directory: {DRIVE_CKPT}')
print(f'Data zip candidates: {len(candidates)}')
for c in candidates[:5]:
    print(f'  - {c}')
assert DRIVE_DATA_ZIP and os.path.exists(DRIVE_DATA_ZIP), (
    'LOB data zip not found under /content/drive/MyDrive/. '
    'Either move drive-download-*.zip to your MyDrive or set DRIVE_DATA_ZIP manually.'
)
print(f'Using: {DRIVE_DATA_ZIP}')

In [ ]:
# Step 3 -- Clone repository (from your fork / branch)
REPO_URL = 'https://github.com/Ruuudy1/FinDrama'
BRANCH = 'colab-cleanup'   # change to whatever branch has the LOB work pushed

import os, shutil
if os.path.exists('/content/Drama'):
    shutil.rmtree('/content/Drama')
!git clone --branch {BRANCH} {REPO_URL} /content/Drama

# Fail fast if the LOB modules are missing (means the branch doesn't have the work)
for required in ['/content/Drama/src/lob/backtester/data_loader.py',
                 '/content/Drama/src/envs/lob_features.py',
                 '/content/Drama/src/sub_models/lob_encoder.py',
                 '/content/Drama/src/train_lob.py',
                 '/content/Drama/src/config_files/configure_lob.yaml']:
    assert os.path.exists(required), f'Missing from repo: {required}. Push the LOB branch first.'
print('LOB modules present.')

In [ ]:
# Step 4 -- System dependencies
!apt-get install -y ffmpeg libsm6 libxext6 -q

In [ ]:
# Step 5 -- Verify PyTorch
import torch
print(f'Torch: {torch.__version__}')
print(f'CUDA:  {torch.cuda.is_available()}')
print(f'GPU:   {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Step 6 -- Install Python dependencies (causal-conv1d + mamba-ssm need --no-build-isolation)
!pip install -q packaging
!pip install causal-conv1d --no-build-isolation -q
!pip install mamba-ssm --no-build-isolation -q
!pip install -q \
    yacs==0.1.8 \
    tensorboardX==2.6.2.2 \
    moviepy==1.0.3 \
    colorama==0.4.6 \
    einops==0.8.0 \
    tqdm \
    "opencv-python-headless>=4.10.0" \
    "gymnasium[atari]" \
    kornia==0.7.3 \
    wandb \
    pytorch_warmup==0.1.1 \
    pandas==2.2.2 \
    line_profiler \
    pyarrow
print('All packages installed.')

In [ ]:
# Step 7 -- Compatibility patches (same 4 as the Atari notebook, gymnasium >= 1.0 + causal-conv1d >= 1.4)
path = '/content/Drama/src/envs/my_atari.py'
with open(path) as f:
    txt = f.read()
if 'register_envs' not in txt:
    txt = txt.replace('import gymnasium as gym',
                      'import gymnasium as gym\nimport ale_py\ngym.register_envs(ale_py)', 1)
    with open(path, 'w') as f:
        f.write(txt)
print('my_atari.py              patched')

path = '/content/Drama/src/eval.py'
with open(path) as f:
    txt = f.read()
if 'shape=(image_size, image_size)' not in txt:
    txt = txt.replace('shape=image_size', 'shape=(image_size, image_size)')
    with open(path, 'w') as f:
        f.write(txt)
print('eval.py                  patched')

path = '/content/Drama/src/mamba_ssm/modules/mamba2.py'
with open(path) as f:
    txt = f.read()
if 'self.use_mem_eff_path = False  # patched' not in txt:
    txt = txt.replace('self.use_mem_eff_path = use_mem_eff_path',
                      'self.use_mem_eff_path = False  # patched: causal-conv1d >= 1.4 C API incompatibility')
    with open(path, 'w') as f:
        f.write(txt)
print('mamba2.py                patched')

path = '/content/Drama/src/mamba_ssm/ops/selective_scan_interface.py'
with open(path) as f:
    txt = f.read()
if '    x, conv1d_weight, conv1d_bias, None, None, None, None, True' not in txt:
    txt = txt.replace(
        '    x, conv1d_weight, conv1d_bias, None, None, None, True',
        '    x, conv1d_weight, conv1d_bias, None, None, None, None, True')
    with open(path, 'w') as f:
        f.write(txt)
print('selective_scan_interface.py patched')
print('\nAll patches applied.')

In [ ]:
# Step 8 -- Extract LOB data from Drive zip (~1.3 GB -> ~5 GB on disk)
import os, shutil, subprocess

DATA_ROOT = '/content/Drama/data'
STAGING = f'{DATA_ROOT}/_staging'
os.makedirs(STAGING, exist_ok=True)

print('Unpacking outer zip...')
subprocess.run(['unzip', '-o', DRIVE_DATA_ZIP, '-d', STAGING], check=True)
print('Unpacking validation.tar.zip...')
subprocess.run(['unzip', '-o', f'{STAGING}/validation.tar.zip', '-x', '__MACOSX/*', '-d', STAGING], check=True)
print('Unpacking train.tar.zip...')
subprocess.run(['unzip', '-o', f'{STAGING}/train.tar.zip', '-x', '__MACOSX/*', '-d', STAGING], check=True)

os.makedirs(f'{DATA_ROOT}/train', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/validation', exist_ok=True)
print('Extracting validation.tar ...')
subprocess.run(['tar', '-xf', f'{STAGING}/validation.tar', '-C', f'{DATA_ROOT}/validation'], check=True)
print('Extracting train.tar ...')
subprocess.run(['tar', '-xf', f'{STAGING}/train.tar', '-C', f'{DATA_ROOT}/train'], check=True)

shutil.rmtree(STAGING)
!du -sh {DATA_ROOT}/* 2>/dev/null | tail
print('Data ready at', DATA_ROOT)

In [ ]:
# Step 9 -- Pretrain world model on LOB features
# Default: 20k steps on the longest-lived market in data/train.
# Checkpoints write to src/saved_models/lob/LOB/<run_id>/ckpt/world_model.pth.
import os
os.environ['WANDB_MODE'] = 'offline'
%cd /content/Drama/src

!python train_lob.py \
    --data-train /content/Drama/data/train \
    --data-val /content/Drama/data/validation \
    --hours-train 6 \
    --hours-val 1 \
    --Wandb.Init.Mode offline

In [ ]:
# Step 10 -- (Optional) Auto-backup LOB checkpoints to Drive. Run in a second tab.
import glob, os, shutil, time

DRIVE_CKPT = '/content/drive/MyDrive/drama_lob_checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

for cycle in range(200):
    time.sleep(600)
    synced = 0
    for src in glob.glob('/content/Drama/src/saved_models/**/*.pth', recursive=True):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(src))
        shutil.copy2(src, dst)
        synced += 1
    for src in glob.glob('/content/Drama/src/saved_models/**/imagine/*.npy', recursive=True):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(src))
        shutil.copy2(src, dst)
        synced += 1
    for src in glob.glob('/content/Drama/data/normalization.json'):
        shutil.copy2(src, os.path.join(DRIVE_CKPT, 'normalization.json'))
        synced += 1
    print(f'[Backup {cycle + 1}] {synced} file(s) synced to Drive')